In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-pinecone langchain-google-genai
!pip install -U pinecone pypdf tiktoken gdown

In [ ]:
!mkdir -p pdfs
!gdown 1hPQlXrX8FbaYaLypxTmeVOFNitbBMlEE -O pdfs/yolov7paper.pdf
!gdown 1vILwiv6nS2wI3chxNabMgry3qnV67TxM -O pdfs/rachelgreecv.pdf

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from pinecone import Pinecone as PineconeClient, ServerlessSpec
from getpass import getpass
import os
import sys
import time

In [ ]:
loader = PyPDFDirectoryLoader("pdfs")
data = loader.load()
data

In [ ]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
text_chunks = text_spliter.split_documents(data)
text_chunks

In [ ]:
len(text_chunks), text_chunks[1], text_chunks[2]

In [ ]:
if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
results = embeddings.embed_query("How are you!")
embedding_dim = len(results)
print(f"Embedding dimension: {embedding_dim}")

In [ ]:
if "PINECONE_API_KEY" not in os.environ or not os.environ["PINECONE_API_KEY"]:
    os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API key: ")

pc = PineconeClient(api_key=os.environ["PINECONE_API_KEY"])
index_name = "test"

existing_indexes = [idx["name"] for idx in pc.list_indexes()]
index_existed = index_name in existing_indexes

if not index_existed:
    pc.create_index(
        name=index_name,
        dimension=embedding_dim,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)
    print(f"Created new index '{index_name}'.")
else:
    print(f"Index '{index_name}' already exists.")

In [ ]:
if index_existed:
    docsearch = PineconeVectorStore.from_existing_index(index_name, embeddings)
    print("Loaded existing index.")
else:
    docsearch = PineconeVectorStore.from_texts(
        [t.page_content for t in text_chunks], embeddings, index_name=index_name
    )
    print("Created new index from texts.")

docsearch

In [ ]:
query = "YOLOv7 outperforms which models"
docs = docsearch.similarity_search(query)
docs

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=docsearch.as_retriever())

In [ ]:
query = "YOLOv7 outperforms which models"
response = qa.invoke({"query": query})
print(response["result"])

In [ ]:
while True:
  user_input = input(f"Input Prompt: ")
  if user_input.lower() == 'exit':
    print("Exiting")
    break
  if user_input == '':
    continue

  result = qa({'query': user_input})
  print(f"Answer: {result['result']}")